# Laboratorio 05 — Redes Convolucionales y Recurrentes Aplicadas
### Tecnologías Emergentes (ISO46B) — Semana 5

En este notebook construirás una CNN de clasificación de imágenes (Fashion-MNIST) y una LSTM de pronóstico sobre la serie climática de Huancayo, y experimentarás con sobreajuste, EarlyStopping y ajuste de hiperparámetros.

**Antes de empezar:**
- Si trabajas en Google Colab, sube `pronostico_huancayo.csv` (el de la Semana 4, ya ampliado) a la sesión.
- Fashion-MNIST se descarga automáticamente la primera vez que se usa — no necesitas subir ningún archivo para esa parte.
- Ejecuta las celdas EN ORDEN.


## Parte 1 — Básico: CNN de clasificación de imágenes

### Paso 1.1 — Cargar y explorar Fashion-MNIST

In [ ]:
from tensorflow import keras
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

NOMBRES_CLASES = [
    "Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín",
]

print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)

In [ ]:
fig, ejes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(ejes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(NOMBRES_CLASES[y_train[i]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Paso 1.2 — Normalizar y preparar los datos

In [ ]:
# TU CÓDIGO AQUÍ: normalizar X_train y X_test a [0, 1] (dividir entre 255.0)
X_train = None
X_test = None

# TU CÓDIGO AQUÍ: agregar la dimensión de canal -> reshape(-1, 28, 28, 1)
X_train = None
X_test = None

print("Forma final de X_train:", X_train.shape)

### Paso 1.3 — Construir la arquitectura CNN

Completa la arquitectura: 2 bloques Conv2D+MaxPooling2D (32 y 64 filtros), Flatten, Dense(64), y salida Dense(10) con softmax.

In [ ]:
from tensorflow.keras import layers

modelo_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    # TU CÓDIGO AQUÍ: agrega los 2 bloques Conv2D+MaxPooling2D, Flatten, Dense(64) y la salida
])

modelo_cnn.summary()

### Paso 1.4 — Compilar, entrenar y evaluar

In [ ]:
# TU CÓDIGO AQUÍ: compilar con optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]


In [ ]:
# TU CÓDIGO AQUÍ: entrenar con fit() — epochs=5, batch_size=64, validation_split=0.2
historial = None

In [ ]:
# TU CÓDIGO AQUÍ: evaluar sobre X_test, y_test
perdida_test, precision_test = None, None
print(f"Precisión en test: {precision_test:.4f}")

## Parte 2 — Intermedio: LSTM sobre la serie climática de Huancayo

### Paso 2.1 — Función de ventaneo (windowing)

In [ ]:
import numpy as np
import pandas as pd


def crear_ventanas(serie, ventana):
    """Convierte una serie 1D en pares (X, y) usando una ventana deslizante."""
    X, y = [], []
    # TU CÓDIGO AQUÍ: recorre la serie y construye X (ventana de valores) e y (siguiente valor)
    return np.array(X), np.array(y)

### Paso 2.2 — Cargar la serie y aplicar el ventaneo

In [ ]:
df = pd.read_csv("pronostico_huancayo.csv")
df = df.sort_values("fecha").reset_index(drop=True)
serie = df["temp_max"].values

VENTANA = 5
X, y = crear_ventanas(serie, VENTANA)
X = X.reshape(-1, VENTANA, 1)

print("X:", X.shape, " y:", y.shape)

### Paso 2.3 — Partición secuencial y escalado

**Importante:** no uses train_test_split aleatorio aquí — las series temporales se particionan de forma secuencial.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TU CÓDIGO AQUÍ: partición secuencial 80/20 (sin shuffle)
corte = int(len(X) * 0.8)
X_train, X_test = None, None
y_train, y_test = None, None

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train.reshape(-1, VENTANA)).reshape(-1, VENTANA, 1)
X_test_esc = escalador.transform(X_test.reshape(-1, VENTANA)).reshape(-1, VENTANA, 1)

print("Entrenamiento:", X_train_esc.shape, " Prueba:", X_test_esc.shape)

### Paso 2.4 — Baseline ingenuo

In [ ]:
from sklearn.metrics import mean_absolute_error

# TU CÓDIGO AQUÍ: predicción = último valor de cada ventana de X_test
prediccion_baseline = None
mae_baseline = mean_absolute_error(y_test, prediccion_baseline)
print(f"MAE del baseline ingenuo: {mae_baseline:.3f} °C")

### Paso 2.5 — Construir, compilar y entrenar la LSTM

In [ ]:
modelo_lstm = keras.Sequential([
    layers.Input(shape=(VENTANA, 1)),
    # TU CÓDIGO AQUÍ: agrega layers.LSTM(16, activation="tanh") y la salida Dense(1)
])

# TU CÓDIGO AQUÍ: compilar con optimizer="adam", loss="mse", metrics=["mae"]


In [ ]:
# TU CÓDIGO AQUÍ: entrenar con fit() — epochs=50, batch_size=8, validation_split=0.2
historial_lstm = None

perdida_lstm, mae_lstm = modelo_lstm.evaluate(X_test_esc, y_test, verbose=0)
print(f"MAE de la LSTM en test: {mae_lstm:.3f} °C")
print(f"MAE del baseline en test: {mae_baseline:.3f} °C")

**Pregunta de reflexión:** ¿la LSTM superó al baseline ingenuo? ¿Qué explicación le das, considerando el tamaño del dataset climático?

## Parte 3 — Avanzado: sobreajuste, EarlyStopping y ajuste de hiperparámetros

### Paso 3.1 — Provocar sobreajuste deliberado

In [ ]:
X_pequeno, y_pequeno = X_train[:2000], y_train[:2000] if False else (None, None)
# NOTA: usa las X_train, y_train de Fashion-MNIST (Parte 1), no las de la serie climática.
# TU CÓDIGO AQUÍ: define X_pequeno, y_pequeno como los primeros 2000 ejemplos de Fashion-MNIST


def construir_modelo():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


modelo_sobreajustado = construir_modelo()
# TU CÓDIGO AQUÍ: entrenar SIN EarlyStopping, epochs=25, validation_split=0.3
historial_sin = None

plt.figure(figsize=(6, 4))
plt.plot(historial_sin.history["loss"], label="entrenamiento")
plt.plot(historial_sin.history["val_loss"], label="validación")
plt.title("SIN EarlyStopping")
plt.legend()
plt.show()

### Paso 3.2 — Aplicar EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

modelo_con_es = construir_modelo()
# TU CÓDIGO AQUÍ: crea un EarlyStopping (monitor="val_loss", patience=3, restore_best_weights=True)
parada_temprana = None

# TU CÓDIGO AQUÍ: entrenar CON el callback de EarlyStopping
historial_con = None

print(f"Entrenamiento detenido en la época {len(historial_con.history['loss'])} de 25 programadas.")

### Paso 3.3 — Experimentar con un hiperparámetro

Elige UNO: batch_size, número de filtros, o learning_rate. Compáralo contra el modelo del Paso 3.2.

In [ ]:
# TU CÓDIGO AQUÍ: construye y entrena una variante con un hiperparámetro distinto,
# y compara su precisión en test contra modelo_con_es.


**Documenta aquí (celda Markdown) qué hiperparámetro elegiste, qué valores comparaste, y qué resultado observaste.**